# 07.04 · BM25 & hybrid search

**Chapter:** Embeddings & Vector Search · **Milestone:** M3

## Goals
- [ ] Lexical search with BM25: how term frequency, rarity (IDF) and length normalisation rank documents
- [ ] Find where BM25 beats embeddings and vice versa
- [ ] Combine both with Reciprocal Rank Fusion (RRF) and measure recall@k

## Resources
- [bm25s](https://github.com/xhluca/bm25s) — free
- [Elastic – Practical BM25](https://www.elastic.co/blog/practical-bm25-part-2-the-bm25-algorithm-and-its-variables) — free
- [RRF paper (Cormack et al.)](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) — free

| Lexical option | Notes |
|---|---|
| **bm25s** / rank_bm25 | in-process Python; bm25s is fast (sparse matrices) |
| **Elasticsearch / OpenSearch** | the industry standard; BM25 + vectors + hybrid built in |
| **Postgres full-text search** (`tsvector`, ParadeDB `pg_search`) | in your existing DB |
| **Tantivy / Meilisearch / Typesense** | fast search engines |
| **Learned sparse** (SPLADE, BGE-M3 sparse) | neural model outputs sparse "keyword" vectors |

In [ ]:
import bm25s
import numpy as np

from aieng import rag

docs, X = rag.corpus(), rag.index()
evalset = rag.eval_set()
print(len(docs), "passages |", len(evalset), "eval questions (gold = passages containing the answer string)")

## 1. BM25 in one formula
$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1 (1 - b + b\,|d|/\text{avgdl})}$$
- rare terms (high IDF) matter more; repeated terms help with diminishing returns ($k_1$); long docs are penalised ($b$).

In [ ]:
retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(docs, stopwords="en"))


def bm25_search(query, k=10):
    ids, scores = retriever.retrieve(bm25s.tokenize([query], stopwords="en", show_progress=False), k=k, show_progress=False)
    return [int(i) for i in ids[0]], scores[0]


def dense_search(query, k=10):
    return [pid for pid, _, _ in rag.search(query, k)]


q = "When did Lincoln become president?"
print("BM25 :", [docs[i][:60] for i in bm25_search(q, 3)[0]])
print("dense:", [docs[i][:60] for i in dense_search(q, 3)])

## 2. Where each one wins
BM25 shines on exact tokens: names, IDs, numbers, rare words. Dense shines on paraphrases and vocabulary mismatch.

In [ ]:
probes = [
    "Gettysburg Address",  # exact name
    "Kentucky log cabin birth",  # keyword bag
    "the guy who freed the slaves in America",  # paraphrase, few exact words
    "large flightless bird from Australia",  # description
]
for p in probes:
    print(f"\n{p!r}\n  BM25 : {docs[bm25_search(p, 1)[0][0]][:90]}\n  dense: {docs[dense_search(p, 1)[0]][:90]}")

## 3. Measuring recall@k on the eval set

In [ ]:
def recall_at_k(search_fn, k=5):
    return np.mean([bool(set(search_fn(r["question"], k)) & r["gold"]) for r in evalset])


results = {"BM25": recall_at_k(lambda q, k: bm25_search(q, k)[0]), "dense (bge-small)": recall_at_k(dense_search)}
print(results)

## 4. Hybrid: Reciprocal Rank Fusion
Scores from BM25 and cosine aren't comparable, so fuse **ranks**: $\text{RRF}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$, with $k \approx 60$.
Simple, robust, no tuning. (Alternative: normalise scores and take a weighted sum.)

In [ ]:
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)


def hybrid_search(query, k=5, depth=50):
    return rrf([bm25_search(query, depth)[0], dense_search(query, depth)])[:k]


results["hybrid (RRF)"] = recall_at_k(hybrid_search)
for name, r in results.items():
    print(f"{name:20} recall@5 = {r:.3f}")

Hybrid usually beats either alone, and protects you from embedding failures on IDs/names (07.01 §2). Most production RAG uses it;
Qdrant, Weaviate, Elasticsearch, OpenSearch, LanceDB and others support hybrid natively.

## Exercises
1. Plot recall@k for k in [1, 3, 5, 10, 20] for all three methods.
2. Try a weighted fusion instead of RRF: min-max normalise both score lists and combine `α·dense + (1-α)·bm25`. Sweep α.
3. Find 5 eval questions where hybrid fails. Read them and the gold passages. Is it a retrieval problem or a label problem (our proxy labels!)?

In [ ]:
# your code here